## Required Work

The cleaning in this notebook has two parts:

- Implement the cleaning recommendations you decided to adopt, including any overrides you need (for example, overriding a type assignment from the tool).
- Implement business rules to identify valid data that can move forward to featurization.

Use one method per step and apply them in sequence, as shown in the example. The business rules are shown below


- Identify loans as distressed or not distressed based on loan status and the interpretation of loan status. A non-distressed loan is one that is active and has not missed a payment.
- Identify invalid loans (loan status, bad dates, null first-disbursement dates).
- Drop attributes you do not need.
- Change types for fields that need type conversion, using `df_tags` as the authoritative source.
- Handle missing values for categorical attributes.
- Handle missing values for numeric attributes.

# Explanation of Loan Statuses Based on Gemini Research

This is the loan status description avaialble from Gemini
------------------------------
## 🟢 Active & Good Standing Statuses
These loans are operational and behaving exactly as intended.

* CURRENT ($54,435$ loans): The borrower is paying on time; the account is completely up to date.
* IN DEFERMENT ($100$ loans): The SBA or the lender has granted a temporary, authorized pause on monthly payments (often used during economic hardships or disasters). [1, 2] 

## 🔵 Successfully Closed Statuses
These loans have reached the end of their lifecycle with no losses incurred.

* PREPAID IN FULL ($34,428$ loans): The borrower paid off the entire loan balance early, ahead of the original maturity date.
* PAID IN FULL ($717$ loans): The borrower successfully completed their regular payment schedule and paid off the loan naturally. [1, 3] 

## 🟡 Delinquent Statuses (Past Due)
These loans are actively falling behind on their payments, broken down by exactly how many monthly cycles have been missed.

* 1 MONTH PAST DUE ($150$ loans) to 9 MONTHS PAST DUE ($7$ loans): The borrower has missed sequential monthly payments.
* >9 MONTHS PAST DUE ($12$ loans): Severe long-term delinquency where the borrower has missed 10 or more consecutive payments but the loan hasn't been officially written off yet. [1] 

## 🔴 Distress, Liquidation, & Default Statuses
These statuses indicate that a business has failed to pay, forcing the lender and the SBA to step in to recover funds.

* IN CATCH-UP ($1,061$ loans): The borrower fell behind but has entered into an approved agreement or payment plan to pay off the past-due balance. [1] 
* PURCH(NOT C/O) [Purchased, Not Charged Off] ($597$ loans): The borrower defaulted, so the SBA "purchased" the guaranteed portion of the loan from the commercial bank. However, it is not yet charged off because active collection efforts or asset liquidations are still ongoing. [4, 5] 
* PAID IN FULL (LIQ) [Liquidation] ($669$ loans): The loan was fully paid off, but only because the lender seized and sold off the business's collateral (real estate, equipment, inventory) to cover the debt. [4] 
* CHARGED-OFF ($910$ loans): The borrower defaulted, all collateral has been liquidated, and the SBA has officially written the remaining balance off as an uncollectible loss. [4] 
* ASSET SALE ($2$ loans): The non-performing loan note itself was sold at a discount to an outside debt buyer/investor to get it off the books. [5] 

## ⚪ Non-Origination Statuses
These applications never actually became live loans.

* CANCELED ($14,302$ loans): The loan approval was withdrawn or voided before closing docs were finalized (either by the borrower or the lender).
* NOT FUNDED ($8,261$ loans): The loan process was finalized and approved, but the capital was never successfully disbursed or accepted by the borrower. [3, 6] 


## The Intelligence Gathered

## The Tagged Dictionary
This is the programatic component that has the gathered intelligence and can be used to apply cleaning rules based on abstract rules we specify in the configuration stage.

### Note about drops
The following attribute sets are dropped:
1. The attributes recommended by the tool - but after review
2. The naics description attribute because it is mostly empty
3. The datetime attributes. The rationale for this is that this is a cross-sectional dataset and if I have to use the date information, I need other fields like either the date of last payment or the outstanding balance on the account, perhaps there is some deep business logic to guestimate from the loan-status field, but since this is a demo, I am dropping it rather than including it.
4. Location ID because we are going to use a geo-coding featurization, which is why we have the geographic entity tagging.
5. Paid-in-full and charge-off outcome fields (`paidinfulldate`, `chargeoffdate`, and `grosschargeoffamount`) are explicitly removed as a leakage guard. These attributes reveal the eventual loan outcome and must not be available to the model at training time.


### Loan Condition Definition

Instead of filtering rows out by loan status, derive a new categorical field called `loancondition` so the dataset keeps every record and uses condition as a feature.

| `loancondition` | `loanstatus` examples | Meaning |
| --- | --- | --- |
| `active` | `COMMIT`, `CURRENT`, `IN DEFERMENT` | The loan is live and not yet distressed. |
| `closed` | `PIF`, `PIFLIQ`, `PAID IN FULL`, `PREPAID IN FULL` | The loan has been paid off or otherwise resolved. |
| `distressed` | `CHGOFF`, `1 MONTH PAST DUE`, `IN CATCH-UP`, `PURCH(NOT C/O)` | The loan is delinquent, defaulted, or in loss recovery. |
| `non-origination` | `CANCLD`, `NOTFUND`, `CANCELED`, `NOT FUNDED` | The application never became a live loan. |
| `unknown` | any other status | The status was not recognized by the mapping. |